<a href="https://colab.research.google.com/github/GraceMolina/SQL-Problem-Solving/blob/main/CreatedAndAddDataToATable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# mount google drive to read the dataset
from google.colab import drive
drive.mount('/content/drive')

# read the dataset
import pandas as pd

file_path = '/content/drive/MyDrive/SQL Problem Solving/Module 1/Dataset.xlsx'

data = pd.ExcelFile(file_path)
print(data.sheet_names)

# read sheets
sheets = pd.read_excel(file_path, sheet_name = None)

for name, df in sheets.items():
  print(name, df.shape)
  display(df.head())



In [ ]:
# read the sheets into a SQL database
import sqlite3

conn = sqlite3.connect(':memory:')

for name, df in sheets.items():
  table = name.strip().replace(' ', '_')
  df.to_sql(table, conn, index = False, if_exists = 'replace')

# confirm the tables exist
pd.read_sql('SELECT name FROM sqlite_master WHERE type = "table"', conn)

In [ ]:
# number of users created by date
query1 = """
SELECT DATE(created_timestamp) AS date,
       COUNT(*) AS users_created
FROM users
GROUP BY date
"""
result1 = pd.read_sql(query1, conn)
result1

In [ ]:
# replace all null values in is_admin with false,
# and count the total number of users who are admins
query2 = """
SELECT COALESCE(is_admin, false) AS admin,
       COUNT(*) AS total_users
FROM users
GROUP BY admin
"""
result2 = pd.read_sql(query2, conn)
result2

In [ ]:
# create a table named daily_counts containing two columns:
# order_date and num_orders. The order_date column stores the date,
# while the num_orders column stores the number of orders for a given date
conn.execute('DROP TABLE IF EXISTS daily_counts')
conn.execute("""
CREATE TABLE daily_counts (order_date DATE, num_orders INTEGER)
""")

In [ ]:
# insert data into the table daily_counts
conn.execute("""
INSERT INTO daily_counts (order_date, num_orders)
VALUES ('2025-01-04', 1),
       ('2025-01-06', 3)
""")
conn.commit()

query3 = """
SELECT *
FROM daily_counts
"""
result3 = pd.read_sql(query3, conn)
result3

In [ ]:
# create a table called "days" with a column "dates"
# that lists all dates from December 31, 2024, to January 8, 2025
conn.execute('DROP TABLE IF EXISTS days')
conn.execute("""
CREATE TABLE days (dates DATE)
""")

In [ ]:
# insert data into the table days
conn.execute("""
INSERT INTO days (dates)
VALUES ('2024-12-31'),
       ('2025-01-01'),
       ('2025-01-02'),
       ('2025-01-03'),
       ('2025-01-04'),
       ('2025-01-05'),
       ('2025-01-06'),
       ('2025-01-07'),
       ('2025-01-08')
""")
conn.commit()

query4 = """
SELECT *
FROM days
"""
result4 = pd.read_sql(query4, conn)
result4

In [ ]:
# list the total number of orders for each date
# between January 1, 2025, and January 7, 2025.
query5 = """
SELECT days.dates,
       COALESCE(daily_counts.num_orders, 0.0) AS number_of_orders
FROM days
LEFT JOIN daily_counts
ON days.dates = daily_counts.order_date
WHERE days.dates >= '2025-01-01'
AND days.dates <= '2025-01-07'
"""
result5 = pd.read_sql(query5, conn)
result5

In [ ]:
# determine the date range for the created_timestamp in the users table
query6 = """
SELECT MIN(created_timestamp) AS min_date,
       MAX(created_timestamp) AS max_date
FROM users
"""
result6 = pd.read_sql(query6, conn)
result6

In [ ]:
# list locale segments present in the table called users
# approach 1
query7 = """
SELECT locale,
       COUNT(*)
FROM users
GROUP BY locale
"""
result7 = pd.read_sql(query7, conn)
result7

In [ ]:
# apprach 2
query8 = """
SELECT DISTINCT locale
FROM users
"""
result8 = pd.read_sql(query8, conn)
result8